In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

ds_path = Path(r"C:\Users\sol77\Desktop\ML-For-CV-Robustness\datasets\datasetGTCombined.csv")

Path.is_file(ds_path)

ds = pd.read_csv(ds_path)

shadowParts = ds[ds["illumR"] < 150]
shadowParts

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
333620,0.021819,0.028007,0.029256,1.638,2.241,0.0,180.0,-63.812729,-56.074120,0.0,-19.026,-0.0,0.08
333621,-0.082849,0.103939,0.088035,1.638,2.241,1.0,180.0,-63.812729,-56.074120,0.0,-19.026,-0.0,0.08
333622,-0.100599,0.098665,0.080628,1.638,2.241,2.0,180.0,-63.812729,-56.074120,0.0,-19.026,-0.0,0.08
333623,-0.105408,0.134701,0.109012,1.638,2.241,3.0,180.0,-63.812729,-56.074120,0.0,-19.026,-0.0,0.08
333624,-0.106927,0.144108,0.116267,1.638,2.241,4.0,180.0,-63.812729,-56.074120,0.0,-19.026,-0.0,0.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
667215,0.001704,0.116098,0.116494,1.226,1.498,15.0,180.0,-42.301559,-85.835487,-0.0,-22.204,-180.0,0.26
667216,-0.010579,0.139158,0.136244,1.226,1.498,16.0,180.0,-42.301559,-85.835487,-0.0,-22.204,-180.0,0.26
667217,-0.040078,0.156488,0.144428,1.226,1.498,17.0,180.0,-42.301559,-85.835487,-0.0,-22.204,-180.0,0.26
667218,0.062480,0.111864,0.126774,1.226,1.498,18.0,180.0,-42.301559,-85.835487,-0.0,-22.204,-180.0,0.26


In [2]:
import numpy as np
import pandas as pd

# Select shadowed observations using GREEN illumination
shadowParts = ds[ds["illumG"] < 150].copy()

# Remove invalid GRVI values first
shadowParts = shadowParts[
    shadowParts["grviIn"].between(-1, 1)
].copy()

# Calculate median for each assigned GRVI
median = shadowParts.groupby("grviOut")["grviIn"].transform("median")

# Calculate Median Absolute Deviation (MAD)
deviation = (shadowParts["grviIn"] - median).abs()

mad = deviation.groupby(
    shadowParts["grviOut"]
).transform("median")

# Robust approximation of standard deviation
robust_std = 1.4826 * mad

# Flag unusually distant observations
k = 4

shadowParts["is_outlier"] = (
    (deviation > k * robust_std) &
    (robust_std > 0)
)

# Keep observations that are not outliers
cleanShadows = shadowParts[~shadowParts["is_outlier"]].copy()

In [3]:
before = shadowParts.groupby("grviOut")["grviIn"].agg(
    beforeCount="count",
    beforeMean="mean",
    beforeStd="std"
)

after = cleanShadows.groupby("grviOut")["grviIn"].agg(
    afterCount="count",
    afterMean="mean",
    afterStd="std"
)

comparison = before.join(after)

comparison["removed"] = (
    comparison["beforeCount"] - comparison["afterCount"]
)

comparison["removedPct"] = (
    100 * comparison["removed"] / comparison["beforeCount"]
)

print(comparison.round(5))

         beforeCount  beforeMean  beforeStd  afterCount  afterMean  afterStd  \
grviOut                                                                        
0.08             480    -0.07121    0.06419         478   -0.07231   0.06203   
0.10             908    -0.01105    0.16058         908   -0.01105   0.16058   
0.12             480    -0.00470    0.07888         476   -0.00747   0.07293   
0.14            1148     0.02134    0.09981        1148    0.02134   0.09981   
0.16            1152     0.04428    0.08928        1152    0.04428   0.08928   
0.18             162    -0.13216    0.07311         157   -0.13662   0.06975   
0.20             580     0.01871    0.09108         577    0.01664   0.08660   
0.22            2192     0.05434    0.09088        2191    0.05417   0.09056   
0.24            1388     0.06008    0.10230        1376    0.05724   0.09810   
0.26            8680     0.07818    0.07561        8658    0.07732   0.07370   
0.28            2844     0.06839    0.08